In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Display settings
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style('whitegrid')

# Load preprocessed dataset
print("Loading data...")
df = pd.read_csv('../data/processed/preprocessed_data.csv')

print(f"Total records: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nClass distribution:")
print(df['sentiment_label'].value_counts())
print(f"\nPercentage distribution:")
print(df['sentiment_label'].value_counts(normalize=True) * 100)


Loading data...
Total records: 100,000
Columns: ['id_comment', 'title', 'body', 'created_at', 'rate', 'recommendation_status', 'is_buyer', 'product_id', 'advantages', 'disadvantages', 'likes', 'dislikes', 'seller_title', 'seller_code', 'true_to_size_rate', 'id_product', 'title_fa', 'Rate', 'Rate_cnt', 'Category1', 'Category2', 'Brand', 'Price', 'Seller', 'Is_Fake', 'min_price_last_month', 'sub_category', 'full_text', 'text_length', 'body_length', 'title_length', 'word_count', 'has_advantages', 'has_disadvantages', 'sentiment_label', 'price_change_pct', 'has_product_info', 'product_popularity', 'seller_avg_rate', 'seller_comment_count', 'seller_rate_std', 'seller_total_likes', 'seller_total_dislikes', 'seller_like_ratio', 'full_text_cleaned', 'title_cleaned', 'body_cleaned']

Class distribution:
sentiment_label
positive    67009
neutral     15166
negative     7046
Name: count, dtype: int64

Percentage distribution:
sentiment_label
positive    75.104516
neutral     16.998240
negative    

In [2]:
# ========================================
# STEP 1: Prepare features and target
# ========================================
print("\n" + "="*60)
print("STEP 1: Feature Preparation")
print("="*60)

# Select text column for modeling
text_column = 'full_text_cleaned'

# Select numerical features
numerical_features = [
    'text_length', 'body_length', 'title_length', 'word_count',
    'has_advantages', 'has_disadvantages', 'likes', 'dislikes',
    'price_change_pct', 'product_popularity',
    'seller_avg_rate', 'seller_comment_count', 'seller_rate_std',
    'seller_total_likes', 'seller_total_dislikes', 'seller_like_ratio'
]

# Check for missing values
print(f"\nMissing values in text column: {df[text_column].isna().sum()}")
print(f"Missing values in numerical features:")
print(df[numerical_features].isna().sum())

# Fill missing values in numerical features
df[numerical_features] = df[numerical_features].fillna(0)

print(f"\nText samples:")
for i in range(3):
    print(f"\n{i+1}. [{df.iloc[i]['sentiment_label']}] {df.iloc[i][text_column][:100]}...")



STEP 1: Feature Preparation

Missing values in text column: 50
Missing values in numerical features:
text_length                  0
body_length                  0
title_length                 0
word_count                   0
has_advantages               0
has_disadvantages            0
likes                        0
dislikes                     0
price_change_pct         96483
product_popularity       87376
seller_avg_rate           4483
seller_comment_count      4483
seller_rate_std           5930
seller_total_likes        4483
seller_total_dislikes     4483
seller_like_ratio        10814
dtype: int64

Text samples:

1. [negative] پیشنهاد نمی شود به درد نمیخوره...

2. [nan] بسته بندی بد می تونست به عنوان یه کالای فرهنگی بهتر بسته بندی بشه تجربه جالبی بود برام بسته بندی جال...

3. [nan] برس ریمل بسته بندیش خوب بود کاربرد و کیفیتشم خیلی خوبه من برای روغن زدم به مژه و لیفت ابرو خریدم...


In [3]:
# ========================================
# STEP 2: Handle missing values and prepare data
# ========================================
print("\n" + "="*60)
print("STEP 2: Data Cleaning and Preparation")
print("="*60)

# Remove rows with missing text or sentiment_label
print(f"\nOriginal dataset size: {len(df)}")
df_clean = df.dropna(subset=[text_column, 'sentiment_label']).copy()
print(f"After removing missing text/labels: {len(df_clean)}")
print(f"Removed rows: {len(df) - len(df_clean)}")

# Fill missing numerical features with 0
df_clean[numerical_features] = df_clean[numerical_features].fillna(0)

# Check sentiment distribution
print(f"\nSentiment distribution:")
print(df_clean['sentiment_label'].value_counts())
print(f"\nSentiment percentages:")
print(df_clean['sentiment_label'].value_counts(normalize=True) * 100)

# Prepare features (X) and target (y)
X_text = df_clean[text_column]
X_numerical = df_clean[numerical_features]
y = df_clean['sentiment_label']

print(f"\nFeature shapes:")
print(f"Text features: {X_text.shape}")
print(f"Numerical features: {X_numerical.shape}")
print(f"Target: {y.shape}")

print(f"\nData ready for train-test split!")



STEP 2: Data Cleaning and Preparation

Original dataset size: 100000
After removing missing text/labels: 89178
Removed rows: 10822

Sentiment distribution:
sentiment_label
positive    66973
neutral     15159
negative     7046
Name: count, dtype: int64

Sentiment percentages:
sentiment_label
positive    75.100361
neutral     16.998587
negative     7.901052
Name: proportion, dtype: float64

Feature shapes:
Text features: (89178,)
Numerical features: (89178, 16)
Target: (89178,)

Data ready for train-test split!


In [4]:
# ========================================
# STEP 3: Train-Test Split
# ========================================
print("\n" + "="*60)
print("STEP 3: Train-Test Split (Stratified)")
print("="*60)

from sklearn.model_selection import train_test_split

# Split with stratification to maintain class proportions
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, 
    X_numerical, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"\nTraining set size: {len(X_text_train)}")
print(f"Test set size: {len(X_text_test)}")

print(f"\nTraining set sentiment distribution:")
print(y_train.value_counts())
print(f"\nTest set sentiment distribution:")
print(y_test.value_counts())

print(f"\nTraining set percentages:")
print(y_train.value_counts(normalize=True) * 100)
print(f"\nTest set percentages:")
print(y_test.value_counts(normalize=True) * 100)

print(f"\nTrain-test split completed successfully!")



STEP 3: Train-Test Split (Stratified)

Training set size: 71342
Test set size: 17836

Training set sentiment distribution:
sentiment_label
positive    53578
neutral     12127
negative     5637
Name: count, dtype: int64

Test set sentiment distribution:
sentiment_label
positive    13395
neutral      3032
negative     1409
Name: count, dtype: int64

Training set percentages:
sentiment_label
positive    75.100221
neutral     16.998402
negative     7.901376
Name: proportion, dtype: float64

Test set percentages:
sentiment_label
positive    75.100919
neutral     16.999327
negative     7.899753
Name: proportion, dtype: float64

Train-test split completed successfully!


In [5]:
# ========================================
# STEP 4: TF-IDF Vectorization
# ========================================
print("\n" + "="*60)
print("STEP 4: TF-IDF Vectorization")
print("="*60)

from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# Initialize TF-IDF vectorizer
# max_features: limit to top 5000 features to manage memory
# min_df: ignore terms that appear in less than 5 documents
# max_df: ignore terms that appear in more than 70% of documents
tfidf = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
    ngram_range=(1, 2)  # unigrams and bigrams
)

print("\nFitting TF-IDF on training text...")
X_text_train_tfidf = tfidf.fit_transform(X_text_train)
print(f"Training TF-IDF shape: {X_text_train_tfidf.shape}")

print("\nTransforming test text...")
X_text_test_tfidf = tfidf.transform(X_text_test)
print(f"Test TF-IDF shape: {X_text_test_tfidf.shape}")

print(f"\nVocabulary size: {len(tfidf.vocabulary_)}")
print(f"TF-IDF vectorization completed!")

# Save the vectorizer
print("\nSaving TF-IDF vectorizer...")
joblib.dump(tfidf, '../data/processed/tfidf_vectorizer.pkl')
print("TF-IDF vectorizer saved to: ../data/processed/tfidf_vectorizer.pkl")



STEP 4: TF-IDF Vectorization

Fitting TF-IDF on training text...
Training TF-IDF shape: (71342, 5000)

Transforming test text...
Test TF-IDF shape: (17836, 5000)

Vocabulary size: 5000
TF-IDF vectorization completed!

Saving TF-IDF vectorizer...
TF-IDF vectorizer saved to: ../data/processed/tfidf_vectorizer.pkl


In [9]:
# Check what X_num_train looks like
print("Type of X_num_train:", type(X_num_train))
print("Shape:", X_num_train.shape if hasattr(X_num_train, 'shape') else 'No shape')
print("\nFirst few rows:")
print(X_num_train.head() if hasattr(X_num_train, 'head') else X_num_train[:5])
print("\nData types:")
print(X_num_train.dtypes if hasattr(X_num_train, 'dtypes') else 'No dtypes')


Type of X_num_train: <class 'pandas.core.frame.DataFrame'>
Shape: (71342, 16)

First few rows:
       text_length  body_length  title_length  word_count  has_advantages  \
26564           75           49            25          15               0   
21807           28           28             0           6               0   
86900           34           34             0           7               0   
296             36           24            11           7               0   
92006           37           37             0           6               0   

       has_disadvantages  likes  dislikes  price_change_pct  \
26564                  0      1         0               0.0   
21807                  0      2         0               0.0   
86900                  0      0         0               0.0   
296                    0      0         0               0.0   
92006                  0      0         0               0.0   

      product_popularity  seller_avg_rate  seller_comment_count

In [10]:
# Fix data types - convert object columns to numeric
print("Converting product_popularity to numeric...")
X_num_train['product_popularity'] = pd.to_numeric(X_num_train['product_popularity'], errors='coerce').fillna(0)
X_num_test['product_popularity'] = pd.to_numeric(X_num_test['product_popularity'], errors='coerce').fillna(0)

# Verify all columns are numeric now
print("\nData types after conversion:")
print(X_num_train.dtypes)
print("\nChecking for any remaining non-numeric values...")
print(X_num_train.select_dtypes(include=['object']).columns.tolist())


Converting product_popularity to numeric...

Data types after conversion:
text_length                int64
body_length                int64
title_length               int64
word_count                 int64
has_advantages             int64
has_disadvantages          int64
likes                      int64
dislikes                   int64
price_change_pct         float64
product_popularity       float64
seller_avg_rate          float64
seller_comment_count     float64
seller_rate_std          float64
seller_total_likes       float64
seller_total_dislikes    float64
seller_like_ratio        float64
dtype: object

Checking for any remaining non-numeric values...
[]


In [11]:
from scipy.sparse import csr_matrix, hstack

# Convert numerical features to sparse matrices
X_num_train_sparse = csr_matrix(X_num_train.values)
X_num_test_sparse = csr_matrix(X_num_test.values)

# Combine TF-IDF + numerical features
X_train_combined = hstack([X_text_train_tfidf, X_num_train_sparse])
X_test_combined = hstack([X_text_test_tfidf, X_num_test_sparse])

print(f"X_train_combined shape: {X_train_combined.shape}")
print(f"X_test_combined shape: {X_test_combined.shape}")


X_train_combined shape: (71342, 5016)
X_test_combined shape: (17836, 5016)


In [22]:
from sklearn.metrics import f1_score, classification_report
import time

# Re-train models with additional metrics
results = {}

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear SVM': LinearSVC(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

print("Training models with full metrics...")
for name, model in models.items():
    print(f"\nTraining {name}...")
    start_time = time.time()
    model.fit(X_train_combined, y_train)
    training_time = time.time() - start_time
    
    y_pred = model.predict(X_test_combined)
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    weighted_f1 = f1_score(y_test, y_pred, average='weighted')
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'time': training_time,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'predictions': y_pred
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Macro F1: {macro_f1:.4f}")
    print(f"  Weighted F1: {weighted_f1:.4f}")
    print(f"  Training Time: {training_time:.2f}s")

# Find best model based on accuracy
best_model_name = max(results, key=lambda x: results[x]['accuracy'])
best_model = results[best_model_name]['model']

print(f"\n=== Best Model: {best_model_name} ===")
print(f"Accuracy: {results[best_model_name]['accuracy']:.4f}")
print(f"Macro F1: {results[best_model_name]['macro_f1']:.4f}")
print(f"Weighted F1: {results[best_model_name]['weighted_f1']:.4f}")

# Save the best model
import pickle
with open('../data/processed/best_model_svm.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f"\nBest model saved to '../data/processed/best_model_svm.pkl'")


Training models with full metrics...

Training Logistic Regression...
  Accuracy: 0.7416
  Macro F1: 0.2938
  Weighted F1: 0.6426
  Training Time: 32.48s

Training Linear SVM...
  Accuracy: 0.8079
  Macro F1: 0.5899
  Weighted F1: 0.7641
  Training Time: 31.51s

Training Random Forest...
  Accuracy: 0.8006
  Macro F1: 0.5789
  Weighted F1: 0.7595
  Training Time: 372.14s

=== Best Model: Linear SVM ===
Accuracy: 0.8079
Macro F1: 0.5899
Weighted F1: 0.7641

Best model saved to '../data/processed/best_model_svm.pkl'


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

# Recreate the TF-IDF vectorizer with the same parameters
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.7,
    ngram_range=(1, 2)
)

# Fit on training data
X_tfidf_train = tfidf_vectorizer.fit_transform(X_text_train)

# Save the vectorizer properly
with open('../data/processed/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print("Vectorizer recreated and saved successfully")
print(f"Vocabulary size: {len(tfidf_vectorizer.get_feature_names_out())}")

# Now analyze feature importance
lr_model = results['Logistic Regression']['model']
feature_names = tfidf_vectorizer.get_feature_names_out().tolist() + numerical_features

# Get coefficients for positive class
coef = lr_model.coef_[2]  # Positive class (index 2)

# Get top 20 positive features
top_positive_idx = coef.argsort()[-20:][::-1]
top_positive_features = [(feature_names[i], coef[i]) for i in top_positive_idx]

print("\nTop 20 features for POSITIVE sentiment:")
for feature, weight in top_positive_features:
    print(f"{feature}: {weight:.4f}")

print("\n" + "="*50 + "\n")

# Get coefficients for negative class
coef_neg = lr_model.coef_[0]  # Negative class (index 0)

# Get top 20 negative features
top_negative_idx = coef_neg.argsort()[-20:][::-1]
top_negative_features = [(feature_names[i], coef_neg[i]) for i in top_negative_idx]

print("Top 20 features for NEGATIVE sentiment:")
for feature, weight in top_negative_features:
    print(f"{feature}: {weight:.4f}")


Vectorizer recreated and saved successfully
Vocabulary size: 5000

Top 20 features for POSITIVE sentiment:
seller_avg_rate: 0.2728
seller_rate_std: 0.0959
seller_like_ratio: 0.0513
عالی: 0.0329
has_advantages: 0.0159
خوب: 0.0120
خوبه: 0.0114
عالیه: 0.0095
خیلی: 0.0077
likes: 0.0074
بود: 0.0074
خوب بود: 0.0072
کتاب: 0.0065
عالی بود: 0.0059
text_length: 0.0059
می کنم: 0.0048
خیلی خوبه: 0.0046
خیلی خوب: 0.0040
می: 0.0037
عالی عالی: 0.0037


Top 20 features for NEGATIVE sentiment:
dislikes: 0.0095
title_length: 0.0088
has_disadvantages: 0.0062
اصلا: 0.0058
price_change_pct: 0.0047
body_length: 0.0034
likes: 0.0031
افتضاح: 0.0029
نبود: 0.0026
نیست: 0.0024
بی: 0.0020
بد: 0.0020
بی کیفیت: 0.0020
نمی: 0.0019
خوب نبود: 0.0018
خوب نیست: 0.0018
مرجوع: 0.0017
نداره: 0.0017
نمی کنم: 0.0015
نداشت: 0.0015


In [24]:
print("Available keys in results['Linear SVM']:")
print(results['Linear SVM'].keys())


Available keys in results['Linear SVM']:
dict_keys(['model', 'accuracy', 'time', 'macro_f1', 'weighted_f1', 'predictions'])
